# Maize Yield Transfer Learning — Training
USA → Indonesia / Vietnam / Thailand
Model: CropYieldLSTM (hidden=256, 2 layers)


In [ ]:
# Fix numpy compatibility issue
!pip install numpy==1.26.4 --force-reinstall --no-deps -q
!pip install pandas --force-reinstall -q

import os
os.kill(os.getpid(), 9)   # Restart kernel otomatis

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 105.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 93.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 17.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
ydata-profiling 4.18.1 requires numpy<2.4,>=1.22, but you have numpy 2.4.4 which is incompatible.
ydata-profiling 4.18.1 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.2 which is incompatible.
google-colab 1.0.0 requires ipykernel==6.17.1, but you have ipykernel 7.2.0 which is incompatible

In [12]:
# 1. Clone repo (depth=1: latest commit only, ~5MB)
import os
!git clone https://github.com/alisulas/thesis-maize-v2.git /kaggle/working/thesis_maize
os.chdir('/kaggle/working/thesis_maize')
!pip install -r requirements.txt -q

fatal: destination path '/kaggle/working/thesis_maize' already exists and is not an empty directory.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 77.7 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.27.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires ipykernel==6.17.1, but you have ipykernel 7.2.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2

In [13]:
  import os, shutil, glob

  # Cari path file npz di /kaggle/input/
  npz_files = glob.glob('/kaggle/input/**/*.npz', recursive=True)
  print("Found npz files:")
  for f in npz_files:
      print(f"  {f}")

Found npz files:
  /kaggle/input/datasets/alisulashidayat/maize-yield-modis-tensors/vnm_modis.npz
  /kaggle/input/datasets/alisulashidayat/maize-yield-modis-tensors/usa_modis.npz
  /kaggle/input/datasets/alisulashidayat/maize-yield-modis-tensors/tha_modis.npz
  /kaggle/input/datasets/alisulashidayat/maize-yield-modis-tensors/idn_modis.npz


In [14]:
# 2. Cell 2 — Copy .npz ke lokasi yang benar:
import os, shutil


src = '/kaggle/input/datasets/alisulashidayat/maize-yield-modis-tensors'
dst = '/kaggle/working/thesis_maize/data/processed/modis'
os.makedirs(dst, exist_ok=True) #folder tujuan


for f in ['usa_modis.npz', 'idn_modis.npz', 'vnm_modis.npz', 'tha_modis.npz']:
  shutil.copy(f'{src}/{f}', f'{dst}/{f}')
  print(f'Copied {f}')

Copied usa_modis.npz
Copied idn_modis.npz
Copied vnm_modis.npz
Copied tha_modis.npz


In [15]:
# Cell 3 — Cek yield data ada di repo:

import os

# Daftar file yang dicek
files_to_check = [
    'data/processed/usa/yield_usa_2003_2023.parquet',
    'data/processed/indonesia/yield_indonesia_province_2020_2024.csv',
    'data/processed/vietnam/yield_vietnam_province_1995_2023.csv',
    'data/processed/thailand/thailand_province_yield_2021_2023.csv',
]

base_path = '/kaggle/working/thesis_maize'

for p in files_to_check:
    full_path = f'{base_path}/{p}'
    exists = os.path.exists(full_path)
    status = "OK" if exists else "MISSING"
    print(f"{status:7} {p}")

OK      data/processed/usa/yield_usa_2003_2023.parquet
OK      data/processed/indonesia/yield_indonesia_province_2020_2024.csv
OK      data/processed/vietnam/yield_vietnam_province_1995_2023.csv
OK      data/processed/thailand/thailand_province_yield_2021_2023.csv


In [16]:
# 3. Verify tensors
import numpy as np
for country in ['usa', 'idn', 'vnm', 'tha']:
    d = np.load(f'data/processed/modis/{country}_modis.npz')
    print(f'{country.upper()}: X={d["X"].shape}, y={d["y"].shape}, yield=[{d["y"].min():.2f}, {d["y"].max():.2f}]')

USA: X=(32296, 46, 10), y=(32296,), yield=[0.00, 16.96]
IDN: X=(162, 46, 10), y=(162,), yield=[0.00, 7.68]
VNM: X=(1315, 46, 10), y=(1315,), yield=[1.48, 9.02]
THA: X=(126, 46, 10), y=(126,), yield=[2.00, 5.57]


In [24]:
import os
os.chdir('/kaggle/working/thesis_maize')
!python src/training/train.py --config experiments/configs/usa_lstm.yaml

2026-05-08 13:29:54,064 [INFO] Experiment: usa_baseline_lstm  |  Country: usa  |  Device: cuda
2026-05-08 13:29:55,409 [INFO]   Train: 27910  Val: 2831  Test: 1349
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 Tesla P100-PCIE-16GB which is of cuda capability 6.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the Tesla 

In [26]:
import shutil, os

ckpt = 'experiments/checkpoints/usa_lstm/best_model.pt'
print("Exists:", os.path.exists(ckpt))

if os.path.exists(ckpt):
  size = os.path.getsize(ckpt) / 1024 / 1024
  print(f"Size: {size:.1f} MB")
  os.makedirs('/kaggle/working/outputs', exist_ok=True)
  shutil.copy(ckpt, '/kaggle/working/outputs/usa_lstm_best.pt')
  print("Copied to outputs/")

Exists: False


In [23]:
# 5. Fine-tune all ASEAN countries
!python src/transfer/finetune.py --country all

2026-05-08 13:26:35,783 [ERROR] Pretrained checkpoint not found: /kaggle/working/thesis_maize/experiments/checkpoints/usa_lstm/best_model.pt
2026-05-08 13:26:35,783 [ERROR] Run: python src/training/train.py --config experiments/configs/usa_baseline.yaml


In [2]:
import os

# Cek current working directory
print("Current directory:", os.getcwd())

# Cari file transfer_results.csv di seluruh folder
print("\nMencari file transfer_results.csv ...")
for root, dirs, files in os.walk('.'):
    for file in files:
        if file == 'transfer_results.csv':
            print(f"Ditemukan: {os.path.join(root, file)}")

Current directory: /kaggle/working

Mencari file transfer_results.csv ...
✅ Ditemukan: ./thesis_maize/experiments/logs/transfer_results.csv


In [3]:
# 6. Show final results
import pandas as pd
# Path yang benar
results_path = './thesis_maize/experiments/logs/transfer_results.csv'
#results_path = pd.read_csv('experiments/logs/transfer_results.csv')

results = pd.read_csv(results_path)

print(results.to_string(index=False))

country  transfer_r2  transfer_rmse  scratch_r2  scratch_rmse  r2_improvement
    idn     0.058958       1.134808   -0.297135      1.332326        0.356093
    vnm    -0.185487       1.458558   -0.091879      1.399789       -0.093608
    tha    -0.026449       0.436365   -0.034280      0.438026        0.007831


In [6]:
import pandas as pd
log = pd.read_csv('./thesis_maize/experiments/logs/usa_baseline_lstm_train_log.csv')
print(log.tail(5)[['epoch','train_loss','val_loss','val_r2']].to_string(index=False))

 epoch  train_loss  val_loss   val_r2
   103    1.754717  3.598251 0.438931
   104    1.745352  3.200180 0.503587
   105    1.715271  3.513285 0.453533
   106    1.719700  3.115678 0.514183
   107    1.714173  3.168822 0.506346


In [10]:
import glob, os

# Cari semua file .pt
pt_files = glob.glob('/kaggle/working/**/*.pt', recursive=True)
print("PT files:", pt_files)

# Cek isi checkpoints folder lebih detail
for root, dirs, files in os.walk('/kaggle/working/thesis_maize/experiments/checkpoints'):
  print(f"DIR: {root}")
  for f in files:
      print(f"  {f}")

# Cek transfer_results.csv ukuran sebenarnya
path = '/kaggle/working/thesis_maize/experiments/logs/transfer_results.csv'
print(f"\ntransfer_results size: {os.path.getsize(path)} bytes")
print(open(path).read())

PT files: []
DIR: /kaggle/working/thesis_maize/experiments/checkpoints
  .gitkeep

transfer_results size: 380 bytes
country,transfer_r2,transfer_rmse,scratch_r2,scratch_rmse,r2_improvement
idn,0.0589576959609985,1.1348079442977903,-0.2971351146697998,1.3323256969451904,0.3560928106307983
vnm,-0.1854866743087768,1.4585583209991455,-0.0918787717819213,1.399789333343506,-0.0936079025268554
tha,-0.026449203491210938,0.4363648593425751,-0.03428041934967041,0.4380262792110443,0.007831215858459473



In [8]:
# Buat folder output
os.makedirs('/kaggle/working/outputs', exist_ok=True)

# Copy checkpoint model USA (yang paling penting!)
src_ckpt = './thesis_maize/experiments/checkpoints/usa_lstm/best_model.pt'
if os.path.exists(src_ckpt):
  shutil.copy(src_ckpt, '/kaggle/working/outputs/usa_lstm_best.pt')
  print(f"Copied checkpoint: {os.path.getsize(src_ckpt)/1024/1024:.1f} MB")

# Copy semua CSV logs
for csv in ['usa_baseline_lstm_train_log.csv', 'transfer_results.csv',
          'finetune_idn_log.csv', 'finetune_vnm_log.csv', 'finetune_tha_log.csv']:
  src = f'./thesis_maize/experiments/logs/{csv}'
  if os.path.exists(src):
      shutil.copy(src, f'/kaggle/working/outputs/{csv}')
      print(f"Copied: {csv}")

print("\nDone! Download dari tab Output → outputs/")

Copied: usa_baseline_lstm_train_log.csv
Copied: transfer_results.csv
Copied: finetune_idn_log.csv
Copied: finetune_vnm_log.csv
Copied: finetune_tha_log.csv

Done! Download dari tab Output → outputs/


In [9]:
# 7. Copy outputs to /kaggle/working for download
import shutil, glob
os.makedirs('/kaggle/working/outputs', exist_ok=True)

# Checkpoints
for ckpt_dir in glob.glob('experiments/checkpoints/*'):
    shutil.copytree(ckpt_dir, f'/kaggle/working/outputs/{os.path.basename(ckpt_dir)}', dirs_exist_ok=True)

# Logs
for csv in glob.glob('experiments/logs/*.csv'):
    shutil.copy(csv, '/kaggle/working/outputs/')

print('Output files:')
for f in sorted(glob.glob('/kaggle/working/outputs/**/*', recursive=True)):
    if os.path.isfile(f):
        print(f'  {f}')

Output files:
  /kaggle/working/outputs/finetune_idn_log.csv
  /kaggle/working/outputs/finetune_tha_log.csv
  /kaggle/working/outputs/finetune_vnm_log.csv
  /kaggle/working/outputs/transfer_results.csv
  /kaggle/working/outputs/usa_baseline_lstm_train_log.csv
